In [0]:

%sql SHOW DATABASES;

SELECT current_catalog(), current_schema();

In [0]:
%sql
SHOW CATALOGS;
SHOW SCHEMAS IN rearc_daily_weather_observations_noaa;

In [0]:
%sql SHOW TABLES IN rearc_daily_weather_observations_noaa.esg_noaa_ghcn

In [0]:
%sql
DESCRIBE default.rma_county_yields_report_399;
DESCRIBE rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily;

In [0]:
%sql
SHOW TABLES IN rearc_daily_weather_observations_noaa.esg_noaa_ghcn;

In [0]:
%sql
DESCRIBE default.rma_county_yields_report_399;
DESCRIBE rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW yields_clean AS
SELECT
  *,
  trim(`State Name`) AS state_name,
  trim(`County Name`) AS county_name,
  lpad(cast(cast(`State Code` as int) as string), 2, '0') AS state_fips,
  lpad(cast(cast(`County Code` as int) as string), 3, '0') AS county_fips3,
  concat(
    lpad(cast(cast(`State Code` as int) as string), 2, '0'),
    lpad(cast(cast(`County Code` as int) as string), 3, '0')
  ) AS county_fips,
  cast(`Yield Year` as int) AS year,
  cast(`Yield Amount` as double) AS yield,
  trim(`Commodity Name`) AS commodity,
  trim(`Irrigation Practice Name`) AS irrigation
FROM default.rma_county_yields_report_399;

In [0]:
%sql
SELECT county_fips, trim(`State Name`) AS state, trim(`County Name`) AS county, year, yield
FROM yields_clean
WHERE trim(`State Name`) = 'Alabama' AND trim(`County Name`) = 'Colbert'
ORDER BY year
LIMIT 5;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW weather_clean AS
SELECT
  station,
  date,
  latitude,
  longitude,
  precipitation / 10.0 AS prcp_mm,
  temp_max / 10.0 AS tmax_c,
  temp_min / 10.0 AS tmin_c
FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily;

SELECT * FROM weather_clean LIMIT 10;

In [0]:
%sql
SELECT
  commodity,
  irrigation,
  COUNT(*) AS rows
FROM yields_clean
GROUP BY commodity, irrigation
ORDER BY rows DESC
LIMIT 50;

In [0]:
%sql
SELECT
  irrigation,
  COUNT(*) AS rows
FROM yields_clean
WHERE commodity = 'Corn'
GROUP BY irrigation
ORDER BY rows DESC;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW yields_all AS
SELECT
  county_fips,
  state_name,
  county_name,
  year,
  yield,
  commodity,
  irrigation
FROM yields_clean;

In [0]:
%sql
SELECT COUNT(*) AS duplicate_groups
FROM (
  SELECT county_fips, year, commodity, irrigation, COUNT(*) AS cnt
  FROM yields_all
  GROUP BY county_fips, year, commodity, irrigation
  HAVING cnt > 1
);

In [0]:
%sql
CREATE OR REPLACE TABLE default.yield_anomalies AS
WITH base AS (
  SELECT county_fips, state_name, county_name, year, yield, commodity, irrigation
  FROM yields_all
),

-- compute stats per crop + irrigation + year
stats AS (
  SELECT
    commodity,
    irrigation,
    year,
    avg(yield) AS mu,
    stddev(yield) AS sigma
  FROM base
  GROUP BY commodity, irrigation, year
)

SELECT
  b.*,
  (b.yield - s.mu) / s.sigma AS yield_z,
  abs((b.yield - s.mu) / s.sigma) >= 2 AS is_anomaly
FROM base b
JOIN stats s
  ON b.commodity = s.commodity
 AND b.irrigation = s.irrigation
 AND b.year = s.year;

In [0]:
%sql
SELECT *
FROM default.yield_anomalies
WHERE is_anomaly = true
ORDER BY abs(yield_z) DESC
LIMIT 30;